# Automated V&V demo — `vivarium_testing_utils.automated_validation`

This notebook drives the new `ValidationContext` API against the most
recent psimulate run for `vivarium_csu_alzheimers`. The goal is to
show, in one place, the four moves the framework expects of you:

1. Point at a results directory.
2. Tell it about non-standard measures.
3. Add comparisons.
4. Verify.

Two project-specific wrinkles need handling first.

**Wrinkle 1 — environment.** The cluster's
`vivarium_csu_alzheimers_simulation` env can no longer import
`vivarium_inputs` because `get_draws` is deprecated and the IHME
`rotisserie` dependency is no longer available on PyPI. The validation
interface only needs a handful of helpers from `vivarium_inputs` for
the GBD-source path, which we are not using. We stub those modules
before importing.

**Wrinkle 2 — psimulate layout.** The model spec on disk does not
record an `artifact_path` (psimulate sweeps over ten country artifacts
through `branches.yaml`), and `ValidationContext` insists on one. We
build a small workspace under `/tmp` that mirrors the run, patches the
spec to point at the United States artifact, and pre-filters each
results dataset to that one country plus a handful of draws. Trimming
keeps the SLURM session under its 8 GB cap.

In [1]:
# --- Wrinkle 1: stub vivarium_inputs before the validation interface imports it.

import os, sys, types

os.environ["DISABLE_PANDERA_IMPORT_WARNING"] = "True"


def _install_vivarium_inputs_stubs():
    vi = types.ModuleType("vivarium_inputs")
    utilities = types.ModuleType("vivarium_inputs.utilities")
    interface = types.ModuleType("vivarium_inputs.interface")
    mapping_ext = types.ModuleType("vivarium_inputs.mapping_extension")
    globals_mod = types.ModuleType("vivarium_inputs.globals")

    def _identity(x, *a, **k):
        return x

    for fn in (
        "get_affected_measure_column",
        "scrub_gbd_conventions",
        "split_interval",
        "sort_hierarchical_data",
    ):
        setattr(utilities, fn, _identity)

    def _gbd_disabled(*a, **k):
        raise RuntimeError("GBD-source loading is disabled in this notebook.")

    interface.get_population_structure = _gbd_disabled
    interface.load_standard_data = _gbd_disabled
    mapping_ext.alternative_risk_factors = {}
    globals_mod.DEMOGRAPHIC_COLUMNS = ["location_id", "sex_id", "age_group_id", "year_id"]
    globals_mod.VIVARIUM_COLUMNS = [
        "location", "sex", "age_start", "age_end", "year_start", "year_end",
    ]
    vi.utilities = utilities
    vi.interface = interface
    vi.mapping_extension = mapping_ext
    vi.globals = globals_mod
    vi.get_age_bins = lambda: _gbd_disabled()

    for name, mod in {
        "vivarium_inputs": vi,
        "vivarium_inputs.utilities": utilities,
        "vivarium_inputs.interface": interface,
        "vivarium_inputs.mapping_extension": mapping_ext,
        "vivarium_inputs.globals": globals_mod,
    }.items():
        sys.modules[name] = mod


_install_vivarium_inputs_stubs()


In [2]:
# --- Wrinkle 2: build a small workspace mirroring the most recent psimulate run.

import shutil
from pathlib import Path

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import yaml

SRC = Path(
    "/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/"
    "model12.2/batch12/model_spec/2026_02_14_07_00_06"
)
USA_ARTIFACT = (
    "/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/artifacts/"
    "model10.0/united_states_of_america.hdf"
)
DRAWS = [161, 50, 24]                              # the three draws batch12 covers

WORKDIR = Path("/tmp/avd_demo") / SRC.name
shutil.rmtree(WORKDIR.parent, ignore_errors=True)
(WORKDIR / "results").mkdir(parents=True)

# Patched model spec: inject the missing artifact_path so DataLoader can find it.
spec = yaml.safe_load((SRC / "model_specification.yaml").read_text())
spec.setdefault("configuration", {}).setdefault("input_data", {})["artifact_path"] = USA_ARTIFACT
(WORKDIR / "model_specification.yaml").write_text(yaml.safe_dump(spec))

# Filtered parquet copies. Streaming with pyarrow keeps the 3 GB-per-file source
# from being materialised in pandas before the filter runs.
flt = (pc.field("artifact_path") == USA_ARTIFACT) & pc.is_in(
    pc.field("input_draw"), pa.array(DRAWS)
)
for sub in sorted((SRC / "results").iterdir()):
    files = sorted(sub.glob("*.parquet")) if sub.is_dir() else []
    if not files:
        continue
    table = ds.dataset(str(files[0]), format="parquet").to_table(filter=flt)
    out = WORKDIR / "results" / sub.name
    out.mkdir()
    pq.write_table(table, out / "0000.parquet")

print("workspace:", WORKDIR)
print("on-disk size:", sum(p.stat().st_size for p in WORKDIR.rglob("*")) / 1e6, "MB")


workspace: /tmp/avd_demo/2026_02_14_07_00_06
on-disk size: 16.466154 MB


## Construct the context

The constructor reads `model_specification.yaml`, loads the artifact,
discovers parquet result datasets, and pulls age bins from the
artifact. `scenario_columns` declares which columns identify a
psimulate branch on the simulation side. Here only
`intervention.scenario` matters — we filtered to one country during
the workspace build, so `artifact_path` is constant.

In [3]:
from vivarium_testing_utils.automated_validation.interface import ValidationContext

ctx = ValidationContext(
    results_dir=WORKDIR,
    scenario_columns=("scenario",),
)
print(f"location:        {ctx.location}")
print(f"sim outputs:     {len(ctx.get_sim_outputs())} datasets")
print(f"artifact keys:   {len(ctx.get_artifact_keys())} entries")
print()
print("relevant artifact keys:")
for key in ctx.get_artifact_keys():
    if "alzheimers" in key:
        print(" ", key)


location:        United States of America
sim outputs:     15 datasets
artifact keys:   33 entries

relevant artifact keys:
  cause.alzheimers.bbbm_conditional_prevalence
  cause.alzheimers.excess_mortality_rate
  cause.alzheimers.mci_conditional_prevalence
  cause.alzheimers.mci_disability_weight
  cause.alzheimers.mci_to_dementia_transition_rate
  cause.alzheimers.population_incidence_rate
  cause.alzheimers.prevalence
  cause.alzheimers.susceptible_to_bbbm_transition_count
  cause.alzheimers_consistent.bbbm_conditional_prevalence
  cause.alzheimers_consistent.dementia_conditional_prevalence
  cause.alzheimers_consistent.excess_mortality_rate
  cause.alzheimers_consistent.mci_conditional_prevalence
  cause.alzheimers_consistent.ode_errors
  cause.alzheimers_consistent.population_incidence_any
  cause.alzheimers_consistent.population_incidence_dementia
  cause.alzheimers_consistent.prevalence_any
  cause.alzheimers_consistent.susceptible_to_bbbm_transition_count
  cause.alzheimers_dis

## Register a custom measure

The shipped `MeasureMapper` knows about the canonical SI/SIS shapes
(`incidence_rate`, `prevalence`, `excess_mortality_rate`,
`cause_specific_mortality_rate`, `remission_rate`), all of which
expect simulation states named `susceptible_to_<cause>` and
`<cause>`. The Alzheimer's model uses five compartments instead —
susceptible → BBBM-AD → MCI-AD → AD → dead — and reports

* `entity = alzheimers_disease_state` in `deaths`,
* `sub_entity = alzheimers_disease_state` in
  `person_time_alzheimers_disease_and_other_dementias`.

So we subclass `RatioMeasure` to point the `Deaths` and
`StatePersonTime` formatters at the right rows, and register it under
`cause.alzheimers.excess_mortality_rate`. That artifact key already
exists and stores the matching rate, and the prevalence weight key
(`cause.alzheimers.prevalence`) is also there.

In [4]:
from vivarium_testing_utils.automated_validation.data_transformation.measures import (
    RatioMeasure, RateAggregationWeights,
)
from vivarium_testing_utils.automated_validation.data_transformation.formatting import (
    Deaths, StatePersonTime,
)


class ADStateExcessMortalityRate(RatioMeasure):
    '''Deaths in the AD-disease compartment divided by person-time in it.'''

    @property
    def rate_aggregation_weights(self):
        return RateAggregationWeights(
            weight_keys={
                "population": "population.structure",
                "prevalence": f"cause.{self.entity}.prevalence",
            },
            formula=lambda population, prevalence: population * prevalence,
            description="Person-time x prevalence weighted average",
        )

    def __init__(self, cause):
        super().__init__(
            entity_type="cause",
            entity=cause,
            measure="excess_mortality_rate",
            numerator=Deaths("alzheimers_disease_state"),
            denominator=StatePersonTime(
                "alzheimers_disease_and_other_dementias",
                "alzheimers_disease_state",
            ),
        )

    def get_measure_data_from_sim_inputs(self, data):
        return data


ctx.add_new_measure("cause.alzheimers.excess_mortality_rate", ADStateExcessMortalityRate)
ctx.add_comparison(
    "cause.alzheimers.excess_mortality_rate",
    test_source="sim",
    ref_source="artifact",
    test_scenarios={"scenario": "baseline"},
)
print("registered:", list(ctx.comparisons))


2026-05-05 22:24:51.778 | INFO     | vivarium_testing_utils.automated_validation.data_transformation.age_groups:_format_dataframe:587 - Rebinning DataFrame age groups from <vivarium_testing_utils.automated_validation.data_transformation.age_groups.AgeSchema object at 0x7f845db749d0> to <vivarium_testing_utils.automated_validation.data_transformation.age_groups.AgeSchema object at 0x7f845dc39ad0>.


/homes/abie/.conda/envs/vivarium_csu_alzheimers_simulation/lib/python3.11/site-packages/vivarium_testing_utils/automated_validation/data_transformation/age_groups.py:662: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked_series_for_col = result_matrix_for_col.stack(
2026-05-05 22:24:51.956 | INFO     | vivarium_testing_utils.automated_validation.data_transformation.age_groups:_format_dataframe:587 - Rebinning DataFrame age groups from <vivarium_testing_utils.automated_validation.data_transformation.age_groups.AgeSchema object at 0x7f845db71550> to <vivarium_testing_utils.automated_validation.data_transformation.age_groups.AgeSchema object at 0x7f845dbc0bd0>.


/homes/abie/.conda/envs/vivarium_csu_alzheimers_simulation/lib/python3.11/site-packages/vivarium_testing_utils/automated_validation/data_transformation/age_groups.py:662: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked_series_for_col = result_matrix_for_col.stack(


registered: ['cause.alzheimers.excess_mortality_rate']


## Verify

`verify()` runs the fuzzy-checker proportion test under the hood: the
sim's deaths/person-time ratio is compared against the artifact rate,
once overall and once per stratum.

In [5]:
passed = ctx.verify("cause.alzheimers.excess_mortality_rate")
print("passed:", passed)


2026-05-05 22:24:54.934 | INFO     | vivarium_testing_utils.automated_validation.interface:_verify:291 - Comparison cause.alzheimers.excess_mortality_rate passed!


passed: True


## Inspect results

The framework's `get_frame` and `plot_comparison` paths route through
`weighted_average`, which on this artifact raises an index-alignment
error: the rate is stored for one year (2023) but population.structure
spans thirty. The verify path sidesteps that aggregation, so the
underlying TestResult objects are populated and we can read them
directly.

In [6]:
import pandas as pd

cmp = ctx.comparisons["cause.alzheimers.excess_mortality_rate"]["sim_artifact"]

overall = cmp.proportion_test_results["overall"]
print("=== overall ===")
for k, v in overall.to_dict().items():
    print(f"  {k:>25}: {v}")

stratified = cmp.proportion_test_results.get("stratified", {})
total_strat = sum(len(g) for g in stratified.values())
failing = [r for g in stratified.values() for r in g.values() if r.reject_null]
print(f"\nstratified tests: {total_strat}")
print(f"        failing: {len(failing)}")


=== overall ===
                       name: cause.alzheimers.excess_mortality_rate
            name_additional: overall
        observed_proportion: 0.19885034811849386
         observed_numerator: 9243041.0
       observed_denominator: 46482397.88090349
         target_lower_bound: 0.20535872908937114
         target_upper_bound: 0.20535872908937114
               bayes_factor: nan
                reject_null: False
       comparison_to_target: No significant difference
                 confidence: Conclusive

stratified tests: 162
        failing: 0


In [7]:
# A peek at the per-stratum table for the (sex, age_group) slice.

(sex_age_group,) = [k for k in stratified if set(k) == {"sex", "age_group"}]
rows = [r.to_dict() for r in stratified[sex_age_group].values()]
df = pd.DataFrame(rows)[
    ["index_info", "observed_proportion", "target_lower_bound",
     "target_upper_bound", "reject_null", "comparison_to_target"]
]
df = df.rename(columns={
    "observed_proportion": "sim_rate",
    "target_lower_bound": "ref_lo",
    "target_upper_bound": "ref_hi",
})
df["age_group"] = df["index_info"].apply(lambda d: d.get("age_group"))
df["sex"] = df["index_info"].apply(lambda d: d.get("sex"))
df = df.drop(columns=["index_info"]).set_index(["sex", "age_group"]).sort_index()
df.head(15)


sim_rate    ref_lo    ref_hi  reject_null  \
sex    age_group                                              
Female 35 to 39   0.000000  0.000000  0.000000        False   
       40 to 44   0.013205  0.009463  0.009463        False   
       45 to 49   0.048840  0.039562  0.039562        False   
       50 to 54   0.068177  0.065571  0.065571        False   
       55 to 59   0.085872  0.076619  0.076619        False   
       60 to 64   0.104979  0.084059  0.084059        False   
       65 to 69   0.101435  0.096902  0.096902        False   
       70 to 74   0.105740  0.112672  0.112672        False   
       75 to 79   0.148927  0.136869  0.136869        False   
       80 to 84   0.216895  0.185863  0.185863        False   
       85 to 89   0.214299  0.247706  0.247706        False   
       90 to 94   0.188836  0.324603  0.324603        False   
       95 plus    0.261689  0.376866  0.376866        False   
Male   35 to 39   0.000000  0.000000  0.000000        False   
       40 to 44   0.012642  0.008214  0.008214        False   

                       comparison_to_target  
sex    age_group                             
Female 35 to 39   No significant difference  
       40 to 44   No significant difference  
       45 to 49   No significant difference  
       50 to 54   No significant difference  
       55 to 59   No significant difference  
       60 to 64   No significant difference  
       65 to 69   No significant difference  
       70 to 74   No significant difference  
       75 to 79   No significant difference  
       80 to 84   No significant difference  
       85 to 89   No significant difference  
       90 to 94   No significant difference  
       95 plus    No significant difference  
Male   35 to 39   No significant difference  
       40 to 44   No significant difference

## What we showed and what is left

* **Wired up** — workspace, ValidationContext, custom measure
  registration, `add_comparison`, `verify`, raw TestResult inspection.
* **Skipped** — `get_frame` / `plot_comparison` / `generate_results`,
  all of which need `_aggregate_sim_input_stratifications` and that
  step demands the artifact's rate share its index with
  `population × prevalence`. The Alzheimer's artifact stores the rate
  for a single year and the population for thirty, so the merge fails
  on the year level. Two ways out: (a) replicate the rate to all
  population years before construction, or (b) extend the framework's
  weighted-average to broadcast a single-year rate across many
  population years. Either belongs in a follow-up.

The interesting headline number is the overall TestResult — observed
`0.199` vs target `0.205`, no significant difference. The point of
this notebook is not the rate but the shape of the API: four calls,
one verdict, and a path to deepening it incrementally.